In [13]:
from azure.storage.blob import BlobServiceClient
import torch
from transformers import AutoModel, AutoTokenizer
import os
from datasets import Dataset
import pandas as pd

# Azure Storage details
AZURE_STORAGE_CONNECTION_STRING = os.getenv("SONAR_STORAGE_KEY")
CONTAINER_NAME = "results"

# Create BlobServiceClient
blob_service_client = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

# Local folder to store downloaded model
local_model_dir = "/Users/jonasklein/biasintransformers/downloaded_model"
os.makedirs(local_model_dir, exist_ok=True)

# Local folder to store downloaded dataset
local_dataset_dir = "/Users/jonasklein/biasintransformers/downloaded_dataset"
os.makedirs(local_dataset_dir, exist_ok=True)

### Loading in checkpoint model and train dataset

In [ ]:
MODEL_PATH = "bert_good_25/bert_checkpoints/checkpoint-201000"

# List of model files to download
model_files = [
    "config.json",
    "generation_config.json",
    "model.safetensors",
    "optimizer.pt",
    "rng_state.pth",
    "scheduler.pt",
    "special_tokens_map.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "trainer_state.json",
    "training_args.bin",
    "vocab.txt"
]

# Download model files
for file in model_files:
    blob_client = container_client.get_blob_client(f"{MODEL_PATH}/{file}")
    local_file_path = os.path.join(local_model_dir, file)

    with open(local_file_path, "wb") as download_file:
        download_file.write(blob_client.download_blob().readall())
    print(f"Downloaded {file} to {local_file_path}")

In [12]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(local_model_dir)
print(tokenizer)

DATASET_PATH = "bert_good_25/train_dataset"

# List of dataset files to download
dataset_files = [
    "data-00000-of-00001.arrow",
    "dataset_info.json",
    "state.json"
]

# Download dataset files
for file in dataset_files:
    blob_client = container_client.get_blob_client(f"{DATASET_PATH}/{file}")
    local_file_path = os.path.join(local_dataset_dir, file)

    with open(local_file_path, "wb") as download_file:
        download_file.write(blob_client.download_blob().readall())
    print(f"Downloaded {file} to {local_file_path}")

BertTokenizerFast(name_or_path='/Users/jonasklein/biasintransformers/downloaded_model', vocab_size=30000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)
Downloaded data-00000-of-00001.arrow to /Users/jonasklein/biasintransfo

### Checking the train dataset with the tokenizer

In [27]:
# Find the token ids of a certain word with the tokenizer
word = "schoonzus"
word_ids = tokenizer.encode(word, add_special_tokens=False)
print(f"Token ids of '{word}': {word_ids}")

# Decode these token ids back to a word
decoded_word = tokenizer.convert_ids_to_tokens(word_ids)
print(f"Decoded word: {decoded_word}")

Token ids of 'schoonzus': [1986, 2415]
Decoded word: ['schoon', '##zus']


In [ ]:
# Load dataset from the Arrow file
dataset = Dataset.from_file(local_dataset_dir + "/data-00000-of-00001.arrow")

# Get the first example
first_example = dataset[0]

# Print the first example
print(first_example)

df = pd.DataFrame(dataset)
print(df.head())
# Print the size of the dataset
print(len(dataset))

# Decode the first example using the tokenizer, while keeping the tokens separated
decoded_example = tokenizer.convert_ids_to_tokens(first_example["input_ids"])
print(decoded_example)
print(len(decoded_example))

{'input_ids': [2, 2757, 711, 303, 364, 5899, 3766, 263, 235, 924, 1235, 304, 4189, 3229, 243, 1235, 19442, 434, 435, 1694, 292, 1054, 18, 18, 18, 323, 485, 303, 532, 295, 325, 11, 80, 16815, 500, 6500, 16, 1286, 235, 1788, 611, 871, 18, 851, 317, 618, 432, 718, 8275, 1417, 328, 235, 10293, 11, 85, 244, 245, 482, 16, 323, 2935, 317, 303, 760, 247, 2334, 2098, 2894, 18, 411, 317, 7436, 295, 775, 1574, 243, 368, 335, 459, 5895, 243, 2364, 448, 385, 718, 13834, 282, 313, 282, 1213, 380, 435, 1134, 317, 1417, 18, 5071, 317, 247, 7277, 244, 1345, 13712, 16, 323, 278, 500, 485, 392, 8601, 18, 352, 290, 6323, 3976, 297, 295, 18814, 6500, 258, 478, 18, 609, 290, 435, 8941, 292, 18, 1319, 711, 303, 1222, 881, 363, 6807, 290, 16, 235, 956, 4817, 244, 235, 8694, 18, 411, 588, 2657, 539, 258, 596, 297, 235, 4692, 258, 5232, 17, 17, 323, 363, 35, 29438, 878, 1390, 4865, 282, 1042, 18, 609, 734, 247, 4868, 282, 1007, 263, 18, 825, 303, 245, 4333, 244, 235, 5247, 6511, 17, 1573, 2920, 2991, 16, 290, 3

In [ ]:
# Define the target word
word = "hij"

# Get token IDs of the target word
word_ids = tokenizer.encode(word, add_special_tokens=False)
print(f"Token ids of '{word}': {word_ids}")

# Load dataset
dataset = Dataset.from_file(local_dataset_dir + "/data-00000-of-00001.arrow")

# Filter dataset: Keep only examples where input_ids contain word_ids
filtered_examples = [
    example for example in dataset 
    if any(
        example["input_ids"][i:i+len(word_ids)] == word_ids 
        for i in range(len(example["input_ids"]) - len(word_ids) + 1)
    )
]

filtered_df = pd.DataFrame(filtered_examples)

# Print dataset size before and after filtering
print(f"Original dataset size: {len(dataset)}")
print(f"Filtered dataset size: {len(filtered_df)}")

# Print the first few rows of the filtered dataset
print(filtered_df.head())

# Decode the first example using the tokenizer, while keeping the tokens separated
decoded_example = tokenizer.convert_ids_to_tokens(filtered_examples[1]["input_ids"])
print(decoded_example)
print(len(decoded_example))

Token ids of 'hij': [303]
Original dataset size: 19968
Filtered dataset size: 14004
                                           input_ids
0  [2, 2757, 711, 303, 364, 5899, 3766, 263, 235,...
1  [2, 5304, 271, 246, 2859, 27864, 255, 8946, 26...
2  [2, 333, 18875, 451, 939, 4086, 18, 28723, 895...
3  [2, 2081, 275, 247, 611, 1277, 12371, 1457, 83...
4  [2, 1715, 952, 13067, 16, 1511, 374, 235, 4303...
['[CLS]', 'Ber', '##ing', '##de', 'vingers', 'trommel', '##den', 'geduldig', 'op', 'de', 'rand', 'van', 'hete', 'auto', '##dak', '##en', ',', 'armen', 'in', 'witte', 'overhemd', '##en', 'staken', 'uit', 'omlaag', 'gedraaid', '##e', 'raampjes', '.', 'Er', 'lagen', 'kranten', 'over', 'sturen', 'uitgesp', '##reid', '.', 'Stephen', 'stapte', 'met', 'snelle', 'pas', 'door', 'de', 'mensenmassa', "'", 's', ',', 'door', 'lagen', 'gebl', '##è', '##r', 'uit', 'autor', '##adi', '##o', "'", 's', '-', 'reclame', '##de', '##unt', '##jes', ',', 'van', 'energie', 'overlopen', '##de', 'dj', "'", 's', ',', 'n

In [34]:
# Find occurrences of word_ids in input_ids and store their indices
filtered_df["hij_indices"] = [
    [i for i in range(len(example["input_ids"]) - len(word_ids) + 1)
     if example["input_ids"][i:i+len(word_ids)] == word_ids]
    for example in filtered_examples
]

# Print the first few rows of the filtered dataset
print(filtered_df.head())

                                           input_ids  \
0  [2, 2757, 711, 303, 364, 5899, 3766, 263, 235,...   
1  [2, 5304, 271, 246, 2859, 27864, 255, 8946, 26...   
2  [2, 333, 18875, 451, 939, 4086, 18, 28723, 895...   
3  [2, 2081, 275, 247, 611, 1277, 12371, 1457, 83...   
4  [2, 1715, 952, 13067, 16, 1511, 374, 235, 4303...   

                                         hij_indices  
0  [3, 27, 62, 130, 176, 189, 218, 274, 285, 370,...  
1                                         [146, 412]  
2                                         [362, 486]  
3              [57, 60, 96, 124, 135, 408, 441, 497]  
4                [246, 254, 304, 377, 384, 399, 477]  


### Get the embedding for a certain sentence

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(local_model_dir)

# Load model
model = AutoModel.from_pretrained(local_model_dir)

model.eval()

# Test with an example input
text = "The doctor said that the patient had a fever. She recommended taking some medicine to reduce it."
inputs = tokenizer(text, return_tensors="pt")

# Print the tokenized input
print("Tokenized input:", inputs)

# Revert back the inputs variable to text, but showing the token id between brackets for each token
print("Tokenized input (with tokens):", tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze()))

# Perform forward pass to get hidden states
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)  # Enable hidden states output

# Extract last hidden state (before MLM head)
last_hidden_state = outputs.hidden_states[-1]  # Shape: (batch_size, seq_length, hidden_dim)

print("Last hidden layer shape:", last_hidden_state.shape)

Some weights of BertModel were not initialized from the model checkpoint at /Users/jonasklein/biasintransformers/downloaded_model and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenized input: {'input_ids': tensor([[    2,  2494, 16255, 21887,   299,  2437,   238,   955,  2021, 15650,
           317,    67,   934,   298,    18,  8899,  5314,  4575,   246,   152,
          7696,   271,  5687,   138, 24956,   650,  4541, 23617,   138,    75,
           144,    18,     3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Tokenized input (with tokens): ['[CLS]', 'The', 'doctor', 'sa', '##id', 'th', '##at', 'the', 'pat', '##ient', 'had', 'a', 'fe', '##ver', '.', 'She', 'rec', '##ommen', '##de', '##d', 'tak', '##ing', 'som', '##e', 'medici', '##ne', 'to', 'reduc', '##e', 'i', '##t', '.', '[SEP]']
Last hidden layer shape: torch.Size([1, 33, 768])
